# 03 — Feature Engineering

**Milestone 3, Phase A + B.** Today's tickets: `HC-M3-02` (holdout strategy), `HC-M3-03` (experiment configuration). `HC-M3-01` (modeling objective) is doc-only — see [`docs/problem_definition.md`](../docs/problem_definition.md) §7.

**Notebook structure for Milestone 3** (deciding this now so it doesn't drift): three notebooks, one per pipeline stage, not one giant file —

- `03_feature_engineering.ipynb` (this one) — Phase A (foundation) + Phase B (feature engineering)
- `04_modeling.ipynb` — Phase C (preprocessing pipeline) through Phase G (final holdout evaluation)
- `05_error_analysis.ipynb` — Phase H

This is also where reusable logic starts moving out of notebooks and into `src/home_credit_default_risk/` as real, importable, testable modules — notebooks call into it rather than containing the logic inline. Per the instructor note this session is working from: Jupyter is sufficient, but scripts are preferred where there's time, and this is exactly the kind of reusable logic (feature functions, config, pipeline construction) that benefits from being real code with unit tests rather than notebook cells.

## `HC-M3-02` — Holdout strategy

**Decision: reuse the Milestone 1 split as the M3 dev/holdout boundary, rather than drawing a new one.** `data/interim/train_valid_split.csv` already is a stratified, seeded, reproducibility-verified 80/20 split (`HC-M1-05`, `HC-M1-09`). Relabeling it rather than resplitting means there's exactly one holdout definition across the whole project — a fresh split here would raise "why does this differ from Milestone 1" for no benefit, and would throw away an already-verified reproducibility guarantee.

Semantic relabeling for M3 (the underlying row assignment doesn't change):

- M1's **`train`** fold (246,008 rows) → M3's **development** pool — this is what `HC-M3-11`'s 5-fold CV runs inside, and what all feature engineering, preprocessing, and hyperparameter tuning ever touches.
- M1's **`valid`** fold (61,503 rows) → M3's **holdout** — frozen from this point forward. Not explored, not fit on, not used to pick between candidates or hyperparameters. Used exactly once, at `HC-M3-21`.

The config module (`src/home_credit_default_risk/config.py`, `HC-M3-03`) is used here rather than local constants, and the cell below doesn't just load the persisted split — it **recomputes** the split from `config.RANDOM_STATE`/`config.TEST_SIZE` against the current data and asserts the result is identical to what's on disk. That's a real reproducibility check, not a restated claim.

In [2]:
import duckdb
import pandas as pd
from sklearn.model_selection import train_test_split

from home_credit_default_risk import config

con = duckdb.connect(str(config.CACHE_DB), read_only=True)
application_train = con.sql("SELECT SK_ID_CURR, TARGET FROM application_train").df()
con.close()

ids = application_train["SK_ID_CURR"]
y = application_train["TARGET"]

print(f"RANDOM_STATE={config.RANDOM_STATE}, TEST_SIZE={config.TEST_SIZE}")

RANDOM_STATE=42, TEST_SIZE=0.2


In [3]:
recomputed_dev_ids, recomputed_holdout_ids = train_test_split(
    ids,
    test_size=config.TEST_SIZE,
    stratify=y,
    random_state=config.RANDOM_STATE,
)

persisted_split = pd.read_csv(config.SPLIT_PATH)
persisted_dev_ids = persisted_split.loc[
    persisted_split["split"] == "train", "SK_ID_CURR"
]
persisted_holdout_ids = persisted_split.loc[
    persisted_split["split"] == "valid", "SK_ID_CURR"
]

assert set(recomputed_dev_ids) == set(persisted_dev_ids), "Development set mismatch"
assert set(recomputed_holdout_ids) == set(persisted_holdout_ids), "Holdout set mismatch"

print("Recomputed split from config matches the persisted split exactly.")
print(f"Development pool: {len(persisted_dev_ids):,} rows")
print(f"Holdout:           {len(persisted_holdout_ids):,} rows")

Recomputed split from config matches the persisted split exactly.
Development pool: 246,008 rows
Holdout:           61,503 rows


**What's checked below is structural, not relational** — row counts and class balance per pool, which are mechanically guaranteed by `stratify=y` and already known before looking at any feature. This is not "peeking" at the holdout in the sense the project's EDA discipline forbids (feature-vs-target relationships, distributions that would inform imputation/outlier decisions) — it's confirming the split mechanism worked, the same category of check as `HC-M1-05`'s positive-rate confirmation.

In [4]:
y_by_id = y.set_axis(ids)
dev_positive_rate = y_by_id.loc[persisted_dev_ids].mean()
holdout_positive_rate = y_by_id.loc[persisted_holdout_ids].mean()

print(f"Development positive rate: {dev_positive_rate:.4f}")
print(f"Holdout positive rate:     {holdout_positive_rate:.4f}")

Development positive rate: 0.0807
Holdout positive rate:     0.0807


### Summary — HC-M3-02 acceptance criteria

- [x] Holdout split created (reused from `HC-M1-05`, relabeled — not redrawn)
- [x] Stratification applied (`stratify=y`, confirmed identical positive rate in both pools above)
- [x] Random state fixed (`config.RANDOM_STATE = 42`)
- [x] Holdout is never used during tuning (structural convention starting now: `persisted_holdout_ids` is not touched again until `HC-M3-21`)
- [x] Split is reproducible (recomputing from `config` against current data reproduced the persisted split exactly — verified above, not assumed)
- [x] Split strategy documented (this section)

Next: `HC-M3-03` — experiment configuration (already introduced above as `src/home_credit_default_risk/config.py`; closed out properly next).

## `HC-M3-03` — Experiment configuration

Already in use above, not introduced fresh here — `RANDOM_STATE`, `TEST_SIZE`, `N_SPLITS`, and `PRIMARY_METRIC` all live in [`src/home_credit_default_risk/config.py`](../src/home_credit_default_risk/config.py), a real importable module rather than notebook-local constants. That distinction matters for reproducibility going forward: `HC-M3-11`'s cross-validation, `HC-M3-16`/`17`'s hyperparameter search, and any script run outside a notebook all import the same `config`, so "which seed did we use" and "how many folds" have exactly one answer across the whole project, not one per notebook that can silently drift.

Covered by unit tests (`tests/test_config.py`) rather than just asserted — `RANDOM_STATE == 42`, `N_SPLITS == 5`, `PRIMARY_METRIC == "roc_auc"`, and that paths resolve under the project root, so a future edit that accidentally changes one of these fails CI/`pytest`, not silently drifts.

### Summary — HC-M3-03 acceptance criteria

- [x] Random seed centralized (`config.RANDOM_STATE`)
- [x] CV configuration centralized (`config.N_SPLITS`)
- [x] Evaluation metric centralized (`config.PRIMARY_METRIC`)
- [x] Configuration reproducible (unit-tested in `tests/test_config.py`; used and verified live in `HC-M3-02` above)

Next: `HC-M3-04` — feature engineering strategy.